In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Bayesian Analysis: Tb2TiO7 (`emcee`), HEiDi

This tutorial demonstrates a practical two-stage workflow for single-crystal
diffraction analysis with EasyDiffraction.

In the first stage, we run a fast local refinement to obtain a sensible
point estimate and parameter uncertainties. In the second stage, we use
these refined values to define fit bounds and then sample the posterior
distribution with emcee.

The example uses constant-wavelength neutron single-crystal diffraction data
for Tb2TiO7 measured on HEiDi at FRM II.

The goal is not only to obtain a good fit, but also to answer Bayesian
questions such as:

- Which parameter values are most probable?
- How broad are the credible intervals?
- Which parameters are strongly correlated?
- How much uncertainty propagates into the calculated reflection
  intensities?

## 🛠️ Import Library

In [2]:
import easydiffraction as edi

## 📦 Define Project

The project object keeps structures, experiments, fit settings, and
plotting utilities together in a single place. We will build the full
workflow inside this object.

In [3]:
project = edi.Project(name='tbti_heidi_emcee')

In [4]:
project.save_as(dir_path='projects/bayesian-emcee-tbti-heidi')

Saving project 📦 'tbti_heidi_emcee' to '../../../projects/bayesian-emcee-tbti-heidi'


├── 📄 project.edi


├── 📁 structures/


├── 📁 experiments/


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 tbti_heidi_emcee.html


## 🧩 Define Structure

For this example we start from a CIF file describing the Tb2TiO7
pyrochlore structure. Loading the structure from CIF is convenient
because it preserves a realistic starting
model without rebuilding the full structure by hand.

In [5]:
structure_path = edi.download_data('struct-tbti', destination='data')

Getting data...


Data 'struct-tbti': Tb2Ti2O7 (crystal structure)


✅ Data 'struct-tbti' already present at '../../../data/struct-tbti.cif'. Keeping existing.


In [6]:
project.structures.add_from_cif_path(structure_path)

In [7]:
structure = project.structures['tbti']

Render the structure to confirm the pyrochlore model loaded from CIF as
expected before configuring the experiment.

In [8]:
project.display.structure(struct_name='tbti')

Structure 🧩 'tbti' (Atom view type: 'covalent')


## 🔬 Define Experiment

Next we download the measured reflection data, create a neutron
single-crystal experiment, and configure the crystal link,
wavelength, and extinction model.

In [9]:
data_path = edi.download_data('meas-tbti-heidi', destination='data')

Getting data...


Data 'meas-tbti-heidi': Tb2Ti2O7, HEiDi (MLZ)


✅ Data 'meas-tbti-heidi' already present at '../../../data/meas-tbti-heidi.xye'. Keeping existing.


In [10]:
project.experiments.add_from_data_path(
    name='heidi',
    data_path=data_path,
    sample_form='single crystal',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

Data loaded successfully


Experiment 🔬 'heidi'. Number of data points: 220.


In [11]:
experiment = project.experiments['heidi']

Link the crystal structure to the experiment and set its scale factor.

In [12]:
experiment.linked_structure.structure_id = 'tbti'
experiment.linked_structure.scale = 1.0

Set the instrument wavelength and starting extinction parameters.
These values provide the initial experiment description for the local
refinement.

In [13]:
experiment.instrument.setup_wavelength = 0.793

In [14]:
experiment.extinction.mosaicity = 35000
experiment.extinction.radius = 10

## 🚀 Initial Refinement

Before Bayesian sampling, it is useful to run a deterministic fit. This
gives us:

- a good point estimate near the best-fit region,
- uncertainties from the local optimizer,
- a quick check that the model and experiment are configured
  sensibly.

In this tutorial we refine a small set of structural and extinction
parameters while keeping occupancies fixed.

In [15]:
structure.atom_sites['O1'].fract_x.free = True

structure.atom_sites['Ti'].occupancy.free = False
structure.atom_sites['O1'].occupancy.free = False
structure.atom_sites['O2'].occupancy.free = False

structure.atom_sites['Tb'].adp_iso.free = True
structure.atom_sites['Ti'].adp_iso.free = True
structure.atom_sites['O1'].adp_iso.free = True
structure.atom_sites['O2'].adp_iso.free = True

In [16]:
experiment.linked_structure.scale.free = True
experiment.extinction.radius.free = True

We keep using the default LMFIT Levenberg-Marquardt minimizer as a fast local
optimizer. Its main purpose here is to provide a stable starting point
and uncertainty estimates for the Bayesian run.

In [17]:
project.analysis.minimizer.show_supported()

Minimizer types


,,Type,Description
1,,bumps,BUMPS library using the default Levenberg-Marquardt method
2,,bumps (amoeba),BUMPS library with Nelder-Mead simplex method
3,,bumps (de),BUMPS library with differential evolution method
4,,bumps (dream),BUMPS library with DREAM Bayesian sampling
5,,bumps (lm),BUMPS library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,emcee,emcee affine-invariant ensemble Bayesian sampling
8,,lmfit,LMFIT library using the default Levenberg-Marquardt method
9,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
10,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [18]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'heidi' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

│   └── 📄 heidi.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 tbti_heidi_emcee.html


The fit-results display summarizes the locally refined values and their
estimated uncertainties.

In [19]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),4.03
4,🔁 Iterations,59
5,📏 Goodness-of-fit (reduced χ²),12.71
6,"📏 R-factor (Rf, %)",7.67
7,"📏 R-factor squared (Rf², %)",8.12
8,"📏 Weighted R-factor (wR, %)",8.52


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,tbti,atom_site,Tb,adp_iso,Å²,0.5300,0.5319,0.0190,0.36 % ↑
2,tbti,atom_site,Ti,adp_iso,Å²,0.4800,0.4776,0.0311,0.50 % ↓
3,tbti,atom_site,O1,fract_x,,0.3280,0.3280,0.0001,0.00 % ↓
4,tbti,atom_site,O1,adp_iso,Å²,0.4500,0.4504,0.0165,0.09 % ↑
5,tbti,atom_site,O2,adp_iso,Å²,0.2300,0.2375,0.0259,3.25 % ↑
6,heidi,extinction,,radius,μm,10.0000,26.4607,1.0003,164.61 % ↑
7,heidi,linked_structure,,scale,,1.0000,2.9236,0.0488,192.36 % ↑


The correlation plot shows how strongly the refined parameters move
together in the local refinement. The measured-vs-calculated plot shows
how well the refined crystal model reproduces the measured reflection
intensities.

In [20]:
project.display.fit.correlations()

In [21]:
project.display.pattern(expt_name='heidi')

## 🎲 Prepare Sampling

Bayesian samplers require finite bounds for the free parameters. Instead of
setting them manually, we derive them from the uncertainties estimated
in the local refinement.

The helper method `set_fit_bounds_from_uncertainty` centers the bounds
on the current parameter value and expands them by a chosen multiple of
the reported uncertainty.

The default `multiplier` is 4. In this single-crystal tutorial we use
a tighter value of `1.5` to keep the sampling window closer to the
locally refined solution.

Show unset fit bounds before setting them from the local refinement
uncertainties.

In [22]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,tbti,atom_site,Tb,adp_iso,0.53188,0.01900,-inf,inf,Å²
2,tbti,atom_site,Ti,adp_iso,0.47759,0.03113,-inf,inf,Å²
3,tbti,atom_site,O1,fract_x,0.32803,0.00009,-inf,inf,
4,tbti,atom_site,O1,adp_iso,0.45043,0.01652,-inf,inf,Å²
5,tbti,atom_site,O2,adp_iso,0.23747,0.02588,-inf,inf,Å²
6,heidi,extinction,,radius,26.46071,1.00034,-inf,inf,μm
7,heidi,linked_structure,,scale,2.92356,0.04884,-inf,inf,


Set fit bounds for all free parameters using `multiplier=1.5`. In this
tutorial that means the posterior pair plot will later refer to a
`±1.5 × uncertainty` region in its title. To widen the sampling window,
increase the multiplier explicitly.

In [23]:
for param in project.free_parameters:
    param.set_fit_bounds_from_uncertainty(multiplier=1.5)

Displaying the free parameters again is a convenient way to confirm
that the fit bounds have been assigned as expected before launching the
sampler.

In [24]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,tbti,atom_site,Tb,adp_iso,0.53188,0.01900,0.50338,0.56038,Å²
2,tbti,atom_site,Ti,adp_iso,0.47759,0.03113,0.43090,0.52428,Å²
3,tbti,atom_site,O1,fract_x,0.32803,0.00009,0.32789,0.32817,
4,tbti,atom_site,O1,adp_iso,0.45043,0.01652,0.42565,0.47520,Å²
5,tbti,atom_site,O2,adp_iso,0.23747,0.02588,0.19866,0.27629,Å²
6,heidi,extinction,,radius,26.46071,1.00034,24.96020,27.96123,μm
7,heidi,linked_structure,,scale,2.92356,0.04884,2.85030,2.99682,


## 🎲 Run Sampling

We now switch from the local minimizer to the Bayesian emcee sampler.

The settings below are intentionally small so the tutorial runs
quickly. For production analysis you would usually increase the number
of steps and often the burn-in as well. emcee also lets you tune how
walkers are initialized, how many walkers are used, and which proposal
move drives the ensemble.

The `burn` setting is auto-resolved when left unset. Here we override
`steps` with a smaller value to keep the tutorial fast, and the
effective burn-in is recomputed automatically.

In [25]:
project.analysis.minimizer.show_supported()

Minimizer types


,,Type,Description
1,,bumps,BUMPS library using the default Levenberg-Marquardt method
2,,bumps (amoeba),BUMPS library with Nelder-Mead simplex method
3,,bumps (de),BUMPS library with differential evolution method
4,,bumps (dream),BUMPS library with DREAM Bayesian sampling
5,,bumps (lm),BUMPS library with Levenberg-Marquardt method
6,,dfols,DFO-LS library for derivative-free least-squares optimization
7,,emcee,emcee affine-invariant ensemble Bayesian sampling
8,,lmfit,LMFIT library using the default Levenberg-Marquardt method
9,,lmfit (least_squares),LMFIT library with SciPy's trust region reflective algorithm
10,*,lmfit (leastsq),LMFIT library with Levenberg-Marquardt least squares method


In [26]:
project.analysis.minimizer.type = 'emcee'

⚠️ Switching minimizer type removes these settings:                                                                               
   • max_iterations                                                                                                               


⚠️ Switching minimizer type adds these settings with defaults:                                                                    
   • burn_in_steps=1000                                                                                                           
   • initialization_method='ball'                                                                                                 
   • parallel_workers=0                                                                                                           
   • population_size=32                                                                                                           
   • proposal_moves='de'                                                                                                          
   • random_seed=None                                                                                                             
   • sampling_steps=5000                                                           

Current minimizer changed to


emcee


In [27]:
project.analysis.minimizer.sampling_steps = 500  # lower than the default 3000
project.analysis.minimizer.burn_in_steps = 100  # lower than the default 600
project.analysis.minimizer.population_size = 16  # lower than the default 32
project.analysis.minimizer.random_seed = 42  # fixed seed for reproducible output

In [28]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'heidi' for 'single' fitting


🚀 Starting fit process with 'emcee'...


📈 Bayesian sampling progress:


...

├── 📁 analysis/


│   ├── 📄 analysis.edi


│   └── 📄 mcmc.h5


└── 📁 reports/


    └── 📄 tbti_heidi_emcee.html


## 📊 Inspect Results

The fit-results display now includes sampler settings, convergence
diagnostics, committed parameter values, and posterior summary
statistics.

In [29]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,sampling_steps,500,Total sampler iterations per chain.
2,burn_in_steps,100,Sampler iterations discarded as warm-up.
3,thinning_interval,1,Sampler thinning interval.
4,population_size,16,Number of chains or walkers.
5,parallel_workers,0,Worker count; 0 uses all available CPUs.
6,initialization_method,ball,emcee walker initialization method.
7,random_seed,42,Random seed; None uses a system-derived seed.
8,proposal_moves,de,Single emcee proposal move; move mixtures are not persisted in v1.


📋 Bayesian fit results:


,Metric,Value
1,🧪 Sampler,emcee
2,❌ Overall status,failed
3,💬 Engine message,emcee sampling completed
4,⏱️ Fitting time (seconds),374.46
5,📏 Goodness-of-fit (reduced χ²),12.72
6,"📏 R-factor (Rf, %)",7.66
7,"📏 R-factor squared (Rf², %)",8.12
8,"📏 Weighted R-factor (wR, %)",8.56
9,📉 Best log-posterior,-1354.33
10,📊 Convergence status,failed


📈 Committed parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,tbti,atom_site,Tb,adp_iso,Å²,0.5319,0.5305,0.0051,0.26 % ↓
2,tbti,atom_site,Ti,adp_iso,Å²,0.4776,0.4731,0.0082,0.95 % ↓
3,tbti,atom_site,O1,fract_x,,0.3280,0.3280,0.0000,0.00 % ↑
4,tbti,atom_site,O1,adp_iso,Å²,0.4504,0.4510,0.0046,0.13 % ↑
5,tbti,atom_site,O2,adp_iso,Å²,0.2375,0.2376,0.0072,0.04 % ↑
6,heidi,extinction,,radius,μm,26.4607,26.3492,0.2748,0.42 % ↓
7,heidi,linked_structure,,scale,,2.9236,2.9203,0.0128,0.11 % ↓


📊 Posterior distribution:


,datablock,category,entry,parameter,units,median,95% CI,r-hat,ess bulk
1,tbti,atom_site,Tb,adp_iso,Å²,0.5319,"[0.5218, 0.5417]",1.073,394.9
2,tbti,atom_site,Ti,adp_iso,Å²,0.4767,"[0.4612, 0.4944]",1.053,384.8
3,tbti,atom_site,O1,fract_x,,0.3280,"[0.3280, 0.3281]",1.056,405.1
4,tbti,atom_site,O1,adp_iso,Å²,0.4506,"[0.4413, 0.4590]",1.036,380.4
5,tbti,atom_site,O2,adp_iso,Å²,0.2376,"[0.2241, 0.2515]",1.044,418.0
6,heidi,extinction,,radius,μm,26.4726,"[25.9703, 27.0238]",1.058,384.0
7,heidi,linked_structure,,scale,,2.9244,"[2.8991, 2.9503]",1.041,399.3


The correlation and posterior-pair plots are complementary:

- `plot_param_correlations` summarizes pairwise structure in a compact
  matrix.
- `plot_posterior_pairs` shows marginal densities on the diagonal and
  posterior contours off-diagonal. In this tutorial its title also
  reminds you that the display region follows the `±1.5 × uncertainty`
  bounds defined above, while numeric subplot ranges are omitted to
  keep the grid readable.

In [30]:
project.display.fit.correlations()

In [31]:
project.display.posterior.pairs()

The one-dimensional posterior distributions below make it easier to
inspect individual parameters in isolation, including asymmetry or
multimodality.

In [32]:
project.display.posterior.distribution()

Finally, the posterior predictive plot propagates the sampled
parameter uncertainty into the calculated single-crystal reflection
intensities.

In [33]:
project.display.posterior.predictive(expt_name='heidi')